# NeuroSeg V2 — single-cell pipeline
Use a private Kaggle notebook with GPU and Internet enabled. No manually attached dataset is necessary if a verified mirror is accessible. Run the following cell (or Save & Run All) once. It downloads/validates the source, preprocesses, trains a fixed 15-epoch baseline, evaluates once on the held-out test set, and packages real results. Do not tune on test results. All results depend on actual execution.

In [ ]:
import subprocess, sys
from pathlib import Path
import torch
import shutil

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."
repo = Path("/kaggle/working/neuroseg_code")
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "neuroseg-v2",
                    "https://github.com/Manz203/brain-tumor-segmentation.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)

# Kaggle already provides PyTorch, torchvision and the GPU runtime.
# Install only missing dependencies, rather than replacing PyTorch.
import importlib.util
required = {"h5py":"h5py", "sklearn":"scikit-learn", "pandas":"pandas", "PIL":"Pillow", "matplotlib":"matplotlib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

out = Path("/kaggle/working/neuroseg_results")
command = [sys.executable, "-u", "-m", "neuroseg.train",
           "--epochs", "15", "--batch", "8", "--out", str(out)]
# Resume an interrupted run in the same session, if both checkpoints exist.
if (out / "last_unet.pth").is_file() and (out / "best_unet.pth").is_file():
    command += ["--resume", str(out / "last_unet.pth")]
subprocess.run(command, cwd=repo, check=True)
assert (out / "test_metrics.json").is_file(), "Evaluation results missing"
assert (out / "best_unet.pth").is_file(), "Best checkpoint missing"
# Keep only the best checkpoint and actual report artifacts in the portable ZIP.
bundle = Path("/kaggle/working/neuroseg_submission")
bundle.mkdir(exist_ok=True)
for p in out.iterdir():
    if p.is_file() and p.name != "last_unet.pth":
        shutil.copy2(p, bundle / p.name)
archive = shutil.make_archive("/kaggle/working/neuroseg_submission", "zip", root_dir=bundle)
print("RESULTS ZIP:", archive)
print((out / "test_metrics.json").read_text())